# Klasik Görüntü İşleme ile Hücre Segmentasyonu, Sayımı ve Zamansal Analizi
**Veri seti:** Cell Tracking Challenge – Fluo-N2DH-GOWT1 (eğitim arşivi, sekans 01 ve 02)

### Çalıştırma sırası ve commit noktaları
1. **Bölüm 0–3:** Kurulum, veri indirme, veri inceleme, seed ile kare seçimi → `selected_frames.csv`
   - 🟢 *Commit 1:* "Veri seçimi"
2. **Bölüm 4–8:** Dev/val referansları, pipeline, kontrollü deneyler, parametre taramaları, doğrulama
   - 🟢 *Commit 2:* "Deneyler ve doğrulama"
3. **Bölüm 9:** Nihai pipeline'ın kilitlenmesi → `final_pipeline.json`, `test_predictions_locked.csv`
   - 🔒 *Commit 3:* "Nihai pipeline ve test tahminleri kilitlendi" (**test referansları henüz okunmadı**)
4. **Bölüm 10–13:** Test değerlendirmesi, zamansal analiz, şekiller, hata analizi
   - 🟢 *Commit 4:* "Test değerlendirmesi"
5. **Bölüm 14:** README, requirements.txt, .gitignore

> **Test sızıntısı koruması:** Sekans 02'nin referans sayıları yalnızca `read_gt_count()` ile okunur ve bu fonksiyon `test_predictions_locked.csv` oluşmadan sekans 02 için hata verir.

> **GitHub'a yüklemeden önce:** *Düzenle → Tüm çıktıları temizle* yapın. Veri seti görüntüsü içeren şekiller `figures_report/` klasörüne yazılır ve `.gitignore` ile repoya girmez.

## 0. Kurulum

In [ ]:
import os, glob, json, time, random, zipfile, urllib.request, platform
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import cv2
import scipy
from scipy import ndimage
import skimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DATA_URL  = "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip"
ZIP_PATH  = "Fluo-N2DH-GOWT1.zip"
DATA_ROOT = "data"
BASE_DIR  = os.path.join(DATA_ROOT, "Fluo-N2DH-GOWT1")

FIG_DIR     = "figures"         # yalnızca grafikler (veri seti görüntüsü YOK) -> GitHub'a gider
FIG_IMG_DIR = "figures_report"  # veri seti görüntüsü içeren şekiller -> yalnızca rapor, GitHub'a GİTMEZ
LOCK_FILE   = "test_predictions_locked.csv"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(FIG_IMG_DIR, exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 110

print("Python      :", platform.python_version())
for name, mod in [("numpy", np), ("pandas", pd), ("opencv", cv2), ("scipy", scipy),
                  ("scikit-image", skimage), ("matplotlib", matplotlib)]:
    print(f"{name:12s}:", mod.__version__)

## 1. Veri setinin indirilmesi

In [ ]:
if not os.path.isdir(BASE_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Veri seti indiriliyor...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
    print("Arşiv çıkarıldı.")

seq01_imgs = sorted(glob.glob(os.path.join(BASE_DIR, "01", "t*.tif")))
seq02_imgs = sorted(glob.glob(os.path.join(BASE_DIR, "02", "t*.tif")))
assert seq01_imgs and seq02_imgs, f"Görüntüler bulunamadı: {BASE_DIR}"
print(f"Sequence 01: {len(seq01_imgs)} kare | Sequence 02: {len(seq02_imgs)} kare")

def load_image(seq_str, fname):
    return cv2.imread(os.path.join(BASE_DIR, seq_str, fname), cv2.IMREAD_UNCHANGED)

## 2. Veri setinin ön incelemesi (Ödev Bölüm 2)
Her iki sekansın tüm karelerinde boyut, veri tipi ve yoğunluk istatistikleri incelenir. Bu adımda **referans etiketleri okunmaz**.

In [ ]:
def describe_sequence(paths, seq_name):
    rows = []
    for p in paths:
        im = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        rows.append({"sequence": seq_name, "filename": os.path.basename(p),
                     "height": im.shape[0], "width": im.shape[1], "dtype": str(im.dtype),
                     "min": int(im.min()), "max": int(im.max()),
                     "mean": float(im.mean()), "std": float(im.std()),
                     "p50": float(np.percentile(im, 50)), "p99": float(np.percentile(im, 99)),
                     "p99_9": float(np.percentile(im, 99.9))})
    return pd.DataFrame(rows)

df_overview = pd.concat([describe_sequence(seq01_imgs, "01"),
                         describe_sequence(seq02_imgs, "02")], ignore_index=True)
df_overview.to_csv("data_overview.csv", index=False)

print("Boyutlar :", df_overview[["height", "width"]].drop_duplicates().values.tolist())
print("Veri tipi:", df_overview["dtype"].unique().tolist())
print("\nSekans bazında kare istatistiklerinin ortalaması:")
print(df_overview.groupby("sequence")[["min", "max", "mean", "std", "p50", "p99", "p99_9"]].mean().round(2).to_string())

# Histogramlar (yalnızca grafik -> figures/)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, seq, paths in zip(axes, ["01", "02"], [seq01_imgs, seq02_imgs]):
    for p in paths[::30]:
        im = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        ax.hist(im.ravel(), bins=128, range=(0, 255), histtype="step", log=True,
                label=os.path.basename(p))
    ax.set_title(f"Sequence {seq} – örnek kare histogramları (log)", fontsize=10, fontweight="bold")
    ax.set_xlabel("Piksel yoğunluğu"); ax.set_ylabel("Piksel sayısı (log)")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig0a_histograms_seq01_seq02.png", dpi=300); plt.show()

# Örnek görüntüler (veri seti görüntüsü -> figures_report/)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, seq, paths in zip(axes, ["01", "02"], [seq01_imgs, seq02_imgs]):
    im = cv2.imread(paths[len(paths) // 2], cv2.IMREAD_UNCHANGED)
    ax.imshow(im, cmap="gray", vmin=0, vmax=np.percentile(im, 99.8))
    ax.set_title(f"Sequence {seq} – {os.path.basename(paths[len(paths)//2])} (görüntüleme için gerilmiş)", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_IMG_DIR}/fig0b_example_frames.png", dpi=300); plt.show()

## 3. Geliştirme, doğrulama ve test karelerinin seçimi (Ödev Bölüm 3)
- **Seed:** öğrenci numarasının son 5 hanesi **+** 2026 (aritmetik toplam olarak yorumlanmıştır).
- **Zamansal boşluk:** iki kare arasındaki fark `> 5` olmalıdır (yani arada en az 5 kare bulunur).
- Seçim sırası ödevdeki gibidir: önce 5 doğrulama, sonra 10 geliştirme karesi, sonra 3 test bloğu.
- Koşullar sağlanmazsa aynı `random.Random(seed)` üreteciyle yeniden çekilir (deterministik tekrar).
- **Bu dosyada referans hücre sayıları yer almaz.**

In [ ]:
STUDENT_ID_LAST5 = 12022
SEED = STUDENT_ID_LAST5 + 2026
MIN_GAP = 5          # |i - j| > MIN_GAP
N_VAL, N_DEV, N_BLOCKS, BLOCK_LEN = 5, 10, 3, 5

def select_dataset_frames(n_seq01, n_seq02, seed):
    rng = random.Random(seed)
    attempts_devval = 0
    while True:
        attempts_devval += 1
        val_indices = sorted(rng.sample(range(n_seq01), N_VAL))
        dev_candidates = [i for i in range(n_seq01)
                          if all(abs(i - v) > MIN_GAP for v in val_indices)]
        if len(dev_candidates) >= N_DEV:
            dev_indices = sorted(rng.sample(dev_candidates, N_DEV))
            break
    attempts_test = 0
    while True:
        attempts_test += 1
        starts = sorted(rng.sample(range(n_seq02 - (BLOCK_LEN - 1)), N_BLOCKS))
        if all(starts[k + 1] - (starts[k] + BLOCK_LEN - 1) > MIN_GAP for k in range(N_BLOCKS - 1)):
            test_blocks = [(b, s + o) for b, s in enumerate(starts, 1) for o in range(BLOCK_LEN)]
            break
    return dev_indices, val_indices, test_blocks, attempts_devval, attempts_test

dev_idx, val_idx, test_blk, att_dv, att_t = select_dataset_frames(len(seq01_imgs), len(seq02_imgs), SEED)

# Koşulların doğrulanması
assert not set(dev_idx) & set(val_idx)
assert all(abs(d - v) > MIN_GAP for d in dev_idx for v in val_idx)
starts = sorted({s for b, s in test_blk if (b, s - 1) not in test_blk})
assert all(starts[k + 1] - (starts[k] + BLOCK_LEN - 1) > MIN_GAP for k in range(len(starts) - 1))

records  = [{"group": "Development", "block": 0, "sequence": "01", "frame_idx": i,
             "filename": os.path.basename(seq01_imgs[i]), "seed": SEED} for i in dev_idx]
records += [{"group": "Validation", "block": 0, "sequence": "01", "frame_idx": i,
             "filename": os.path.basename(seq01_imgs[i]), "seed": SEED} for i in val_idx]
records += [{"group": "Test", "block": b, "sequence": "02", "frame_idx": i,
             "filename": os.path.basename(seq02_imgs[i]), "seed": SEED} for b, i in test_blk]

df_selected = pd.DataFrame(records)
df_selected.to_csv("selected_frames.csv", index=False)

print(f"Seed = {STUDENT_ID_LAST5} + 2026 = {SEED} | deneme sayısı: dev/val={att_dv}, test={att_t}")
print("\nTABLO 1 – Seçilen kareler")
print(df_selected.to_string(index=False))

## 4. Referans hücre sayıları (Ödev Bölüm 4)
`XX_GT/TRA/man_trackTTT.tif` içindeki sıfır olmayan farklı etiketler sayılır. Bu adımda **yalnızca geliştirme ve doğrulama** kareleri okunur. Sekans 02 (test) kilit dosyası oluşmadan okunamaz.

In [ ]:
def read_gt_labels(seq_str, frame_idx):
    if seq_str == "02" and not os.path.exists(LOCK_FILE):
        raise RuntimeError("Test referansları, test tahminleri kilitlenmeden okunamaz!")
    path = os.path.join(BASE_DIR, f"{seq_str}_GT", "TRA", f"man_track{int(frame_idx):03d}.tif")
    if not os.path.exists(path):
        return None
    return cv2.imread(path, cv2.IMREAD_UNCHANGED)

def read_gt_count(seq_str, frame_idx):
    lbl = read_gt_labels(seq_str, frame_idx)
    return None if lbl is None else int(np.count_nonzero(np.unique(lbl)))

df_devval = df_selected[df_selected["group"].isin(["Development", "Validation"])].copy()
df_devval["gt_count"] = [read_gt_count(r.sequence, r.frame_idx) for r in df_devval.itertuples()]
missing = df_devval[df_devval["gt_count"].isna()]
print("Eksik referans:", "yok" if missing.empty else missing["filename"].tolist())
df_devval.to_csv("dev_val_references.csv", index=False)

dev_df = df_devval[df_devval["group"] == "Development"].reset_index(drop=True)
val_df = df_devval[df_devval["group"] == "Validation"].reset_index(drop=True)
print(df_devval[["group", "frame_idx", "filename", "gt_count"]].to_string(index=False))

# Görsel kontrol: bir geliştirme karesinde TRA işaretçileri (veri seti görüntüsü -> figures_report/)
r0 = dev_df.iloc[0]
im0, gt0 = load_image("01", r0["filename"]), read_gt_labels("01", r0["frame_idx"])
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(im0, cmap="gray", vmin=0, vmax=np.percentile(im0, 99.8))
for g in np.unique(gt0)[1:]:
    ys, xs = np.nonzero(gt0 == g)
    ax.plot(xs.mean(), ys.mean(), "r+", ms=10, mew=2)
ax.set_title(f"{r0['filename']}: {r0['gt_count']} TRA işaretçisi (görsel kontrol)", fontsize=9)
ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_IMG_DIR}/fig0c_gt_visual_check.png", dpi=300); plt.show()

## 5. Pipeline bileşenleri ve baseline (Ödev Bölüm 5–9)
Tek bir `run_pipeline(img, config)` fonksiyonu tüm konfigürasyonları çalıştırır:
**yoğunluk dönüşümü → filtre → segmentasyon → morfoloji → nesne ayırma ve alan filtresi.**
Etiketleme ve alan filtresi vektörize edilmiştir, böylece süre ölçümleri yöntemler arasında adildir.

In [ ]:
HM_REF_FILENAME = dev_df.loc[0, "filename"]          # histogram matching referansı: YALNIZCA geliştirme kümesinden
hm_ref_img = load_image("01", HM_REF_FILENAME)
print("Histogram matching referansı:", HM_REF_FILENAME)

def match_histograms_uint8(source, template):
    _, bin_idx, src_counts = np.unique(source.ravel(), return_inverse=True, return_counts=True)
    tmpl_values, tmpl_counts = np.unique(template.ravel(), return_counts=True)
    src_q = np.cumsum(src_counts).astype(np.float64) / source.size
    tmpl_q = np.cumsum(tmpl_counts).astype(np.float64) / template.size
    return np.interp(src_q, tmpl_q, tmpl_values)[bin_idx].reshape(source.shape).astype(np.uint8)

def filter_by_area(label_img, min_area):
    # label_img içindeki min_area'dan küçük nesneleri siler, kalanları 1..N olarak yeniden etiketler
    areas = np.bincount(label_img.ravel())
    keep = np.nonzero(areas >= min_area)[0]
    keep = keep[keep != 0]
    lut = np.zeros(len(areas), dtype=np.int32)
    lut[keep] = np.arange(1, len(keep) + 1)
    return lut[label_img], len(keep)

def run_pipeline(img, config, return_stages=False):
    t0 = time.perf_counter()
    stages = {}
    proc = img.copy()

    # 1) Yoğunluk dönüşümü
    inten = config.get("intensity", "none")
    if inten == "stretch":
        p_low, p_high = np.percentile(proc, config.get("stretch_p", (1, 99.8)))
        proc = np.clip((proc - p_low) * 255.0 / (p_high - p_low + 1e-5), 0, 255).astype(np.uint8)
    elif inten == "hist_eq":
        proc = cv2.equalizeHist(proc)
    elif inten == "clahe":
        proc = cv2.createCLAHE(clipLimit=config.get("clahe_clip", 3.0),
                               tileGridSize=tuple(config.get("clahe_grid", [8, 8]))).apply(proc)
    elif inten == "hist_match":
        proc = match_histograms_uint8(proc, hm_ref_img)
    stages["intensity"] = proc

    # 2) Uzamsal filtre
    filt, k = config.get("filter", "none"), config.get("filter_k", 3)
    if filt == "gaussian":
        proc = cv2.GaussianBlur(proc, (k, k), 0)
    elif filt == "median":
        proc = cv2.medianBlur(proc, k)
    elif filt == "sharpen":
        kernel_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
        proc = cv2.filter2D(proc, -1, kernel_sharp)
    stages["filtered"] = proc

    # 3) Segmentasyon
    seg = config.get("segmentation", "otsu")
    if seg == "fixed":
        _, binary = cv2.threshold(proc, config.get("fixed_thresh", 20), 255, cv2.THRESH_BINARY)
    elif seg == "otsu":
        _, binary = cv2.threshold(proc, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif seg == "adaptive":
        binary = cv2.adaptiveThreshold(proc, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
                                       config.get("adapt_block", 51), config.get("adapt_c", -5))
    stages["binary"] = binary

    # 4) Morfoloji
    morph, mk = config.get("morph", "none"), config.get("morph_k", 3)
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (mk, mk))
    if morph == "erosion":
        binary = cv2.erode(binary, se)
    elif morph == "dilation":
        binary = cv2.dilate(binary, se)
    elif morph == "opening":
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, se)
    elif morph == "closing":
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se)
    elif morph == "open_close":
        binary = cv2.morphologyEx(cv2.morphologyEx(binary, cv2.MORPH_OPEN, se), cv2.MORPH_CLOSE, se)
    elif morph == "close_open":
        binary = cv2.morphologyEx(cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se), cv2.MORPH_OPEN, se)
    stages["morph"] = binary

    # 5) Nesne ayırma + alan filtresi
    min_area = config.get("min_area", 150)
    _, cc = cv2.connectedComponents(binary, connectivity=8)
    labels, count = filter_by_area(cc.astype(np.int32), min_area)
    if config.get("separation", "cc") == "watershed":
        clean_bin = (labels > 0).astype(np.uint8) * 255
        dist = ndimage.distance_transform_edt(clean_bin)
        coords = peak_local_max(dist, min_distance=config.get("ws_min_dist", 14), labels=clean_bin)
        peaks = np.zeros(dist.shape, dtype=bool)
        peaks[tuple(coords.T)] = True
        markers, _ = ndimage.label(peaks)
        ws = watershed(-dist, markers, mask=clean_bin)
        labels, count = filter_by_area(ws.astype(np.int32), min_area)
        stages["distance"] = dist
    stages["labels"] = labels

    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    return (count, labels, elapsed_ms, stages) if return_stages else (count, labels, elapsed_ms)

def evaluate(df, config, seq_str="01"):
    preds, times = [], []
    for r in df.itertuples():
        cnt, _, dt = run_pipeline(load_image(seq_str, r.filename), config)
        preds.append(cnt); times.append(dt)
    preds, gts = np.array(preds), df["gt_count"].to_numpy()
    err = preds - gts
    return {"MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "Bias": float(np.mean(err)), "Exact_%": float(np.mean(err == 0) * 100),
            "Sure_ms": float(np.mean(times)), "preds": preds.tolist()}

# Isınma turu: ilk çağrıdaki kütüphane yükleme maliyeti süre ölçümlerine girmesin
_w = load_image("01", dev_df.loc[0, "filename"])
for _cfg in [{"segmentation": "otsu"}, {"segmentation": "otsu", "separation": "watershed"}]:
    run_pipeline(_w, _cfg)

## 6. Kontrollü deneyler (Ödev Bölüm 6–10)
Her deney, **referans deneyden yalnızca bir faktörle** ayrılacak şekilde tasarlanmıştır (`Referans` ve `Degisen` sütunları).

| Grup | Deneyler | Karşılaştırma |
|---|---|---|
| Yoğunluk | E01–E05 | orijinal / stretch / HE / CLAHE / HM |
| Filtre | E04, E06–E09 | yok / Gauss 3×3 / Gauss 5×5 / Median 5×5 / keskinleştirme |
| Segmentasyon | E07, E10, E11 | Otsu / sabit / adaptif |
| Morfoloji (tür, SE boyutu) | E07, E12, E15–E19 | erosion, dilation, opening, closing; 3×3 ve 5×5 |
| Morfoloji sırası | E13 vs E20 | close→open / open→close |
| Nesne ayırma | E20 vs E14 | bağlı bileşen / distance transform + watershed |

> Aday pipeline'lar (E13, E11, E14) ilk deney turunda tanımlanmıştır ve değiştirilmemiştir. E12'deki `min_area` 100'e çekilmiş, E15–E20 sonradan eklenmiştir. Bu deneyler yalnızca geliştirme kümesinde çalışır ve aday seçimini etkilemez.

In [ ]:
def make(base, **changes):
    c = dict(base); c.update(changes); return c

E01 = {"intensity": "none", "filter": "none", "segmentation": "otsu", "morph": "none", "separation": "cc", "min_area": 100}
E04 = make(E01, intensity="clahe", clahe_clip=3.0, clahe_grid=[8, 8])
E07 = make(E04, filter="gaussian", filter_k=5)
E13 = make(E07, morph="close_open", morph_k=5, min_area=150)

EXPERIMENTS = [
    # id, açıklama, referans, değişen faktör, konfigürasyon
    ("E01", "Baseline: orijinal + Otsu + CC", "-", "-", E01),
    ("E02", "Contrast stretching (1–99.8%)", "E01", "yoğunluk", make(E01, intensity="stretch", stretch_p=(1, 99.8))),
    ("E03", "Histogram equalization", "E01", "yoğunluk", make(E01, intensity="hist_eq")),
    ("E04", "CLAHE (clip=3.0, 8×8)", "E01", "yoğunluk", E04),
    ("E05", "Histogram matching (ref: dev)", "E01", "yoğunluk", make(E01, intensity="hist_match")),
    ("E06", "CLAHE + Gauss 3×3", "E04", "filtre", make(E04, filter="gaussian", filter_k=3)),
    ("E07", "CLAHE + Gauss 5×5", "E04", "filtre", E07),
    ("E08", "CLAHE + Median 5×5", "E04", "filtre", make(E04, filter="median", filter_k=5)),
    ("E09", "CLAHE + keskinleştirme (3×3 Laplace)", "E04", "filtre", make(E04, filter="sharpen")),
    ("E10", "Sabit eşik T=45", "E07", "segmentasyon", make(E07, segmentation="fixed", fixed_thresh=45)),
    ("E11", "Adaptif Gauss (blok=71, C=-12)", "E07", "segmentasyon", make(E07, segmentation="adaptive", adapt_block=71, adapt_c=-12)),
    ("E12", "Opening 5×5", "E07", "morfoloji", make(E07, morph="opening", morph_k=5)),
    ("E13", "Close→Open 5×5 (min_area=150)", "E07", "morfoloji+alan", E13),
    ("E14", "Open→Close 5×5 + watershed", "E20", "ayırma", make(E07, morph="open_close", morph_k=5, min_area=150, separation="watershed", ws_min_dist=16)),
    ("E15", "Erosion 3×3", "E07", "morfoloji", make(E07, morph="erosion", morph_k=3)),
    ("E16", "Dilation 3×3", "E07", "morfoloji", make(E07, morph="dilation", morph_k=3)),
    ("E17", "Opening 3×3", "E12", "SE boyutu", make(E07, morph="opening", morph_k=3)),
    ("E18", "Closing 3×3", "E07", "morfoloji", make(E07, morph="closing", morph_k=3)),
    ("E19", "Closing 5×5", "E18", "SE boyutu", make(E07, morph="closing", morph_k=5)),
    ("E20", "Open→Close 5×5 + CC (min_area=150)", "E13", "morfoloji sırası", make(E13, morph="open_close")),
]
EXP = {eid: cfg for eid, *_, cfg in EXPERIMENTS}

def param_str(c):
    keys = ["stretch_p", "clahe_clip", "clahe_grid", "filter_k", "fixed_thresh", "adapt_block",
            "adapt_c", "morph_k", "ws_min_dist", "min_area"]
    return ", ".join(f"{k}={c[k]}" for k in keys if k in c)

exp_rows, per_frame = [], []
for eid, desc, ref, changed, cfg in EXPERIMENTS:
    m = evaluate(dev_df, cfg)
    exp_rows.append({"Deney_ID": eid, "Aciklama": desc, "Referans": ref, "Degisen": changed,
                     "Yogunluk": cfg["intensity"], "Filtre": cfg["filter"], "Segmentasyon": cfg["segmentation"],
                     "Morfoloji": cfg["morph"], "Ayirma": cfg["separation"], "Parametreler": param_str(cfg),
                     "Dev_MAE": round(m["MAE"], 3), "Dev_RMSE": round(m["RMSE"], 3),
                     "Dev_Bias": round(m["Bias"], 3), "Sure_ms": round(m["Sure_ms"], 1)})
    for r, p in zip(dev_df.itertuples(), m["preds"]):
        per_frame.append({"Deney_ID": eid, "filename": r.filename, "gt_count": r.gt_count, "pred": p})

df_exp = pd.DataFrame(exp_rows)
df_exp.to_csv("experiments.csv", index=False)
pd.DataFrame(per_frame).to_csv("experiments_per_frame.csv", index=False)
print("TABLO 2 – Geliştirme kümesi deney sonuçları")
print(df_exp[["Deney_ID", "Aciklama", "Referans", "Degisen", "Dev_MAE", "Dev_RMSE", "Dev_Bias", "Sure_ms"]].to_string(index=False))

# Deney MAE grafiği (yalnızca grafik -> figures/)
fig, ax = plt.subplots(figsize=(11, 3.8))
plot_df = df_exp.copy()
plot_df["Dev_MAE_kirpilmis"] = plot_df["Dev_MAE"].clip(upper=20)
ax.bar(plot_df["Deney_ID"], plot_df["Dev_MAE_kirpilmis"], color="#1E88E5")
for i, v in enumerate(plot_df["Dev_MAE"]):
    ax.text(i, min(v, 20) + 0.3, f"{v:.1f}", ha="center", fontsize=7.5)
ax.set_ylabel("Geliştirme MAE (20'de kırpıldı)")
ax.set_title("Kontrollü deneylerin geliştirme kümesi MAE değerleri", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig_experiments_mae.png", dpi=300); plt.show()

## 7. Parametre duyarlılık analizi (yalnızca geliştirme kümesi)
Seçilen parametre değerlerinin çevresinde tek parametre değiştirilerek MAE'nin nasıl değiştiği incelenir. Bu analiz **aday pipeline'ları değiştirmez**. Amacı, seçilen değerlerin ne kadar duyarlı olduğunu raporlamaktır.

In [ ]:
SWEEPS = [
    ("E04", "clahe_clip", [1.0, 2.0, 3.0, 4.0, 6.0]),
    ("E10", "fixed_thresh", [25, 35, 45, 55, 65]),
    ("E11", "adapt_c", [-5, -8, -12, -15, -20]),
    ("E11", "adapt_block", [31, 51, 71, 91, 111]),
    ("E11", "min_area", [50, 100, 150, 200]),
    ("E14", "ws_min_dist", [8, 12, 16, 20, 24]),
]
sweep_rows = []
for base_id, param, values in SWEEPS:
    for v in values:
        m = evaluate(dev_df, make(EXP[base_id], **{param: v}))
        sweep_rows.append({"Taban_deney": base_id, "Parametre": param, "Deger": v,
                           "Secilen_deger": EXP[base_id].get(param) == v,
                           "Dev_MAE": round(m["MAE"], 3), "Dev_Bias": round(m["Bias"], 3)})
df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv("parameter_sweeps.csv", index=False)
print(df_sweep.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))
for ax, (base_id, param, _) in zip(axes.ravel(), SWEEPS):
    s = df_sweep[(df_sweep["Taban_deney"] == base_id) & (df_sweep["Parametre"] == param)]
    ax.plot(s["Deger"], s["Dev_MAE"], "o-", color="#1565C0")
    sel = s[s["Secilen_deger"]]
    ax.plot(sel["Deger"], sel["Dev_MAE"], "o", ms=11, mfc="none", mec="#C62828", mew=2, label="seçilen")
    ax.set_title(f"{base_id}: {param}", fontsize=9.5, fontweight="bold")
    ax.set_xlabel(param); ax.set_ylabel("Dev MAE"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.suptitle("Parametre duyarlılık analizi (geliştirme kümesi)", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig_parameter_sweeps.png", dpi=300); plt.show()

## 8. Doğrulama ve nihai pipeline seçimi (Ödev Bölüm 11)
Üç aday pipeline 5 doğrulama karesinde karşılaştırılır. Seçim **otomatik** yapılır: en düşük MAE; eşitlikte RMSE, süre ve karmaşıklık (adım sayısı).

In [ ]:
CANDIDATES = {
    "Pipeline A (CLAHE+Gauss5+Otsu+Close→Open+CC)": "E13",
    "Pipeline B (CLAHE+Gauss5+Adaptif+CC)": "E11",
    "Pipeline C (CLAHE+Gauss5+Otsu+Open→Close+Watershed)": "E14",
}
def complexity(c):
    return sum([c["intensity"] != "none", c["filter"] != "none", 1,
                c["morph"] != "none", c["separation"] == "watershed"])

val_rows = []
for name, eid in CANDIDATES.items():
    m = evaluate(val_df, EXP[eid])
    val_rows.append({"Pipeline": name, "Deney_ID": eid, "Validation_MAE": round(m["MAE"], 3),
                     "Validation_RMSE": round(m["RMSE"], 3), "Validation_Bias": round(m["Bias"], 3),
                     "Ortalama_Sure_ms": round(m["Sure_ms"], 1), "Karmasiklik": complexity(EXP[eid]),
                     "Val_Tahminler": m["preds"], "Val_Gercek": val_df["gt_count"].tolist()})
df_val = pd.DataFrame(val_rows)
ranked = df_val.sort_values(["Validation_MAE", "Validation_RMSE", "Ortalama_Sure_ms", "Karmasiklik"])
FINAL_NAME = ranked.iloc[0]["Pipeline"]
FINAL_ID = CANDIDATES[FINAL_NAME]
df_val["Secim"] = np.where(df_val["Pipeline"] == FINAL_NAME, "SEÇİLDİ (nihai)", "elendi")
df_val.to_csv("validation_results.csv", index=False)

print("TABLO 3 – Aday pipeline doğrulama sonuçları")
print(df_val.drop(columns=["Val_Tahminler", "Val_Gercek"]).to_string(index=False))
print(f"\nNihai pipeline: {FINAL_NAME} ({FINAL_ID}) – {param_str(EXP[FINAL_ID])}")

## 9. 🔒 Nihai pipeline'ın kilitlenmesi ve test tahminleri (Ödev Bölüm 12)
Bu hücre **test referanslarını okumaz**. Tahminler `test_predictions_locked.csv` dosyasına yazılır. Dosya zaten varsa yeniden üretilen tahminlerin onunla birebir aynı olduğu doğrulanır.

> ⛔ **Bu hücreden sonra durun ve GitHub'a commit atın:** `final_pipeline.json` + `test_predictions_locked.csv` + notebook. Bölüm 10'a ancak bundan sonra geçin.

In [ ]:
BASELINE_CFG = EXP["E01"]
FINAL_CFG = EXP[FINAL_ID]

with open("final_pipeline.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED,
               "baseline": {"experiment_id": "E01", "config": BASELINE_CFG},
               "final_pipeline": {"name": FINAL_NAME, "experiment_id": FINAL_ID, "config": FINAL_CFG,
                                  "selection": "en düşük doğrulama MAE (eşitlikte RMSE, süre, karmaşıklık)",
                                  "validation": ranked.iloc[0][["Validation_MAE", "Validation_RMSE", "Ortalama_Sure_ms"]].to_dict()},
               "histogram_matching_reference": HM_REF_FILENAME},
              f, indent=2, ensure_ascii=False, default=float)

test_frames = df_selected[df_selected["group"] == "Test"].reset_index(drop=True)   # gt_count sütunu YOK
locked_rows = []
for r in test_frames.itertuples():
    img = load_image("02", r.filename)
    b_cnt, _, b_ms = run_pipeline(img, BASELINE_CFG)
    f_cnt, _, f_ms = run_pipeline(img, FINAL_CFG)
    locked_rows.append({"block": r.block, "sequence": "02", "frame_idx": r.frame_idx, "filename": r.filename,
                        "baseline_pred": b_cnt, "final_pred": f_cnt,
                        "baseline_time_ms": round(b_ms, 2), "final_time_ms": round(f_ms, 2)})
df_new_lock = pd.DataFrame(locked_rows)

if os.path.exists(LOCK_FILE):
    df_old = pd.read_csv(LOCK_FILE)
    same = (df_old[["frame_idx", "baseline_pred", "final_pred"]].to_numpy()
            == df_new_lock[["frame_idx", "baseline_pred", "final_pred"]].to_numpy()).all()
    assert same, "UYARI: Yeniden üretilen test tahminleri kilitli dosyadan farklı!"
    print("Kilitli dosya mevcut; yeniden üretilen tahminler birebir aynı. Dosyaya dokunulmadı.")
else:
    df_new_lock.to_csv(LOCK_FILE, index=False)
    print(f"{LOCK_FILE} yazıldı. ⛔ Şimdi commit atın, sonra Bölüm 10'a geçin.")
print(df_new_lock.to_string(index=False))

## 10. Bağımsız test performansı (Ödev Bölüm 13)
Test referansları **ilk kez burada** okunur. Tahminler kilitli dosyadan alınır, yeniden hesaplanmaz.

In [ ]:
df_test = pd.read_csv(LOCK_FILE)
df_test.insert(4, "gt_count", [read_gt_count("02", i) for i in df_test["frame_idx"]])
df_test["baseline_abs_err"] = (df_test["baseline_pred"] - df_test["gt_count"]).abs()
df_test["final_abs_err"] = (df_test["final_pred"] - df_test["gt_count"]).abs()
df_test.to_csv("test_results.csv", index=False)

def summarize(pred, gt, times):
    err = pred - gt
    return {"MAE": round(err.abs().mean(), 3), "RMSE": round(float(np.sqrt((err ** 2).mean())), 3),
            "Bias": round(err.mean(), 3), "Exact_Accuracy_%": round((err == 0).mean() * 100, 1),
            "Ort_Sure_ms": round(times.mean(), 2)}

df_summary = pd.DataFrame([
    {"Yontem": "Baseline (E01)", **summarize(df_test["baseline_pred"], df_test["gt_count"], df_test["baseline_time_ms"])},
    {"Yontem": f"Nihai ({FINAL_ID})", **summarize(df_test["final_pred"], df_test["gt_count"], df_test["final_time_ms"])},
])
df_summary.to_csv("test_summary.csv", index=False)

print("TABLO 4 – Test kareleri")
print(df_test[["block", "frame_idx", "filename", "gt_count", "baseline_pred", "final_pred",
               "baseline_abs_err", "final_abs_err"]].to_string(index=False))
print("\nTABLO 5 – Test özet metrikleri")
print(df_summary.to_string(index=False))

# Doğrulama ve test karşılaştırması (Bölüm 16.6)
v = df_val[df_val["Deney_ID"] == FINAL_ID].iloc[0]
cmp = pd.DataFrame([
    {"Kume": "Doğrulama (Seq 01)", "MAE": v["Validation_MAE"], "RMSE": v["Validation_RMSE"], "Bias": v["Validation_Bias"],
     "Ort_GT": float(np.mean(v["Val_Gercek"]))},
    {"Kume": "Test (Seq 02)", "MAE": df_summary.iloc[1]["MAE"], "RMSE": df_summary.iloc[1]["RMSE"],
     "Bias": df_summary.iloc[1]["Bias"], "Ort_GT": df_test["gt_count"].mean()},
])
cmp.to_csv("validation_vs_test.csv", index=False)
print("\nNihai pipeline: doğrulama ve test karşılaştırması")
print(cmp.round(3).to_string(index=False))

## 11. Zamansal analiz (Ödev Bölüm 14)

In [ ]:
temporal = df_test[["block", "frame_idx", "gt_count", "baseline_pred", "final_pred"]].copy()
for col in ["gt_count", "baseline_pred", "final_pred"]:
    temporal[f"d_{col}"] = temporal.groupby("block")[col].diff()
temporal.to_csv("temporal_analysis.csv", index=False)

stab = temporal.groupby("block").agg(
    GT_ort=("gt_count", "mean"),
    GT_ardisik_degisim=("d_gt_count", lambda s: s.abs().mean()),
    Baseline_ardisik_degisim=("d_baseline_pred", lambda s: s.abs().mean()),
    Nihai_ardisik_degisim=("d_final_pred", lambda s: s.abs().mean()),
    Baseline_std=("baseline_pred", "std"), Nihai_std=("final_pred", "std")).round(2)
print("Bloklara göre ardışık kare sayım değişimi (ortalama |Δ|)")
print(stab.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, b in zip(axes, sorted(df_test["block"].unique())):
    s = df_test[df_test["block"] == b]
    ax.plot(s["frame_idx"], s["gt_count"], "o-", lw=2.2, color="#2E7D32", label="Gerçek (TRA)")
    ax.plot(s["frame_idx"], s["final_pred"], "s--", lw=2, color="#1565C0", label=f"Nihai ({FINAL_ID})")
    ax.plot(s["frame_idx"], s["baseline_pred"], "^:", lw=1.8, color="#C62828", label="Baseline (E01)")
    ax.set_title(f"Test bloğu {b} (kare {s['frame_idx'].min()}–{s['frame_idx'].max()})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Kare numarası"); ax.set_xticks(s["frame_idx"]); ax.grid(ls="--", alpha=0.4)
axes[0].set_ylabel("Hücre sayısı"); axes[-1].legend(fontsize=8.5)
plt.suptitle("Şekil 5. Üç test bloğunda gerçek ve tahmin edilen hücre sayıları", fontsize=11.5, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig5_temporal_blocks.png", dpi=300); plt.show()

## 12. Rapor şekilleri (Ödev Bölüm 17)

In [ ]:
# ---------- Şekil 1: nihai pipeline akış diyagramı (nihai konfigürasyondan otomatik) ----------
def flow_steps(c):
    s = ["Girdi görüntüsü\n(1024×1024, uint8)"]
    names = {"stretch": "Contrast stretching", "hist_eq": "Histogram eşitleme", "clahe": "CLAHE", "hist_match": "Histogram eşleme"}
    if c["intensity"] != "none":
        extra = f"\nclip={c.get('clahe_clip')}, grid={c.get('clahe_grid')}" if c["intensity"] == "clahe" else ""
        s.append(names[c["intensity"]] + extra)
    if c["filter"] != "none":
        s.append({"gaussian": "Gauss filtre", "median": "Median filtre", "sharpen": "Keskinleştirme"}[c["filter"]]
                 + (f"\n{c['filter_k']}×{c['filter_k']}" if "filter_k" in c else ""))
    seg = {"otsu": "Otsu eşikleme", "fixed": f"Sabit eşik\nT={c.get('fixed_thresh')}",
           "adaptive": f"Adaptif Gauss eşik\nblok={c.get('adapt_block')}, C={c.get('adapt_c')}"}
    s.append(seg[c["segmentation"]])
    if c["morph"] != "none":
        s.append(f"Morfoloji: {c['morph']}\nelips {c['morph_k']}×{c['morph_k']}")
    if c["separation"] == "watershed":
        s.append(f"Distance transform\n+ watershed\nmin_distance={c['ws_min_dist']}")
    s.append(f"Bağlı bileşen +\nalan filtresi\n(≥ {c['min_area']} px) → sayım")
    return s

steps = flow_steps(FINAL_CFG)
n = len(steps)
fig, ax = plt.subplots(figsize=(2.4 * n, 2.6)); ax.set_xlim(0, 2.4 * n); ax.set_ylim(0, 2.6); ax.axis("off")
for i, t in enumerate(steps):
    x = 0.2 + 2.4 * i
    ax.add_patch(patches.FancyBboxPatch((x, 0.6), 1.9, 1.4, boxstyle="round,pad=0.08",
                                        facecolor="#E3F2FD", edgecolor="#1565C0", lw=1.8))
    ax.text(x + 0.95, 1.3, t, ha="center", va="center", fontsize=8.5, fontweight="bold")
    if i < n - 1:
        ax.annotate("", xy=(x + 2.38, 1.3), xytext=(x + 1.98, 1.3), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title(f"Şekil 1. Nihai pipeline akış diyagramı – {FINAL_NAME}", fontsize=10.5, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig1_pipeline_flowchart.png", dpi=300); plt.show()

# ---------- Şekil 2: yoğunluk dönüşümleri + histogramlar ----------
ex = dev_df.iloc[0]; img_ex = load_image("01", ex["filename"])
variants = [("Orijinal", E01), ("Contrast stretching (1–99.8%)", EXP["E02"]), ("Histogram eşitleme", EXP["E03"]),
            ("CLAHE (clip=3.0, 8×8)", E04), (f"Histogram eşleme (ref {HM_REF_FILENAME})", EXP["E05"])]
fig, axes = plt.subplots(2, len(variants), figsize=(3.3 * len(variants), 6.4))
for j, (ttl, c) in enumerate(variants):
    cnt, _, _, st = run_pipeline(img_ex, c, return_stages=True)
    im = st["intensity"]
    axes[0, j].imshow(im, cmap="gray", vmin=0, vmax=255); axes[0, j].axis("off")
    axes[0, j].set_title(f"{ttl}\nsayım={cnt} (GT={ex['gt_count']})", fontsize=8.5, fontweight="bold")
    axes[1, j].hist(im.ravel(), bins=64, range=(0, 255), log=True, color="#1E88E5")
    axes[1, j].set_xlabel("Yoğunluk"); axes[1, j].grid(alpha=0.3)
axes[1, 0].set_ylabel("Piksel sayısı (log)")
plt.suptitle(f"Şekil 2. Yoğunluk dönüşümleri ve histogramları ({ex['filename']}, Otsu + CC ile sayım)", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_IMG_DIR}/fig2_intensity_histograms.png", dpi=300); plt.show()

# ---------- Şekil 3: segmentasyon ve morfoloji karşılaştırması ----------
panel_ids = ["E07", "E10", "E11", "E15", "E16", "E17", "E12", "E20", "E14"]
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax, eid in zip(axes.ravel(), panel_ids):
    cnt, lbl, _ = run_pipeline(img_ex, EXP[eid])
    ax.imshow(lbl > 0, cmap="gray"); ax.axis("off")
    desc = next(d for e, d, *_ in EXPERIMENTS if e == eid)
    ax.set_title(f"{eid}: {desc}\nsayım={cnt} (GT={ex['gt_count']})", fontsize=8.5, fontweight="bold")
plt.suptitle(f"Şekil 3. Segmentasyon ve morfolojik işlem çıktıları ({ex['filename']})", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_IMG_DIR}/fig3_seg_morph_comparison.png", dpi=300); plt.show()

# ---------- Şekil 4: nihai pipeline aşama aşama ----------
cnt, lbl, _, st = run_pipeline(img_ex, FINAL_CFG, return_stages=True)
rgb = cv2.cvtColor(st["intensity"], cv2.COLOR_GRAY2RGB)
cnts, _ = cv2.findContours((lbl > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
cv2.drawContours(rgb, cnts, -1, (255, 60, 60), 2)
stage_imgs = [(img_ex, "1. Orijinal"), (st["intensity"], "2. Yoğunluk dönüşümü"), (st["filtered"], "3. Filtre"),
              (st["binary"], "4. Segmentasyon"), (lbl > 0, "5. Morfoloji/ayırma + alan filtresi"),
              (rgb, f"6. Sonuç: {cnt} hücre (GT={ex['gt_count']})")]
fig, axes = plt.subplots(1, len(stage_imgs), figsize=(3 * len(stage_imgs), 3.4))
for ax, (im, t) in zip(axes, stage_imgs):
    ax.imshow(im, cmap=None if np.ndim(im) == 3 else "gray",
              vmax=np.percentile(im, 99.8) if t.startswith("1.") else None)
    ax.set_title(t, fontsize=9, fontweight="bold"); ax.axis("off")
plt.suptitle(f"Şekil 4. Nihai pipeline'ın aşama aşama çıktıları ({ex['filename']})", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_IMG_DIR}/fig4_step_by_step.png", dpi=300); plt.show()

## 13. Hata analizi (Ödev Bölüm 15 ve Şekil 6)
Bu bölüm nihai pipeline'ı **değiştirmez**. Hataların nedenini inceler:
- Her GT hücresi, TRA işaretçisinin altındaki tahmin nesnesine göre üç gruptan birine ayrılır: **doğru**, **birleşmiş** (başka GT hücreleriyle aynı nesnede) veya **kaçırılan** (altında nesne yok).
- GT işaretçisine değmeyen tahmin nesneleri **yanlış pozitif** sayılır.
- Özdeşlik: `GT − Tahmin = kaçırılan + (birleşmiş hücre − birleşmiş nesne) − yanlış pozitif`

In [ ]:
def analyze_frame(seq_str, frame_idx, fname, cfg):
    img = load_image(seq_str, fname)
    pred_count, pred_lbl, _ = run_pipeline(img, cfg)
    gt = read_gt_labels(seq_str, frame_idx)
    cells = []
    for g in np.unique(gt)[1:]:
        m = gt == g
        under = pred_lbl[m]; under = under[under > 0]
        pid = int(np.bincount(under).argmax()) if under.size else 0
        cells.append({"gt_id": int(g), "pred_id": pid, "brightness": float(img[m].mean())})
    hits = pd.Series([c["pred_id"] for c in cells if c["pred_id"] > 0], dtype=int).value_counts()
    for c in cells:
        c["category"] = "kacirilan" if c["pred_id"] == 0 else ("birlesmis" if hits[c["pred_id"]] > 1 else "dogru")
    info = {"gt_count": len(cells), "pred_count": int(pred_count),
            "dogru": sum(c["category"] == "dogru" for c in cells),
            "kacirilan": sum(c["category"] == "kacirilan" for c in cells),
            "birlesmis_hucre": sum(c["category"] == "birlesmis" for c in cells),
            "birlesmis_nesne": int((hits > 1).sum()),
            "yanlis_pozitif": int(pred_count - len(hits))}
    return cells, info, img, pred_lbl, gt

cell_rows, frame_rows = [], []
for split, df_, seq in [("Dogrulama (Seq 01)", val_df, "01"), ("Test (Seq 02)", df_test, "02")]:
    for r in df_.itertuples():
        cells, info, *_ = analyze_frame(seq, r.frame_idx, r.filename, FINAL_CFG)
        for c in cells:
            cell_rows.append({**c, "split": split, "frame_idx": r.frame_idx})
        frame_rows.append({**info, "split": split, "frame_idx": r.frame_idx, "filename": r.filename})
df_cells, df_frames = pd.DataFrame(cell_rows), pd.DataFrame(frame_rows)
df_cells.to_csv("error_analysis_cells.csv", index=False)
df_frames.to_csv("error_analysis_frames.csv", index=False)

chk = (df_frames["gt_count"] - df_frames["pred_count"]) - (df_frames["kacirilan"] + df_frames["birlesmis_hucre"]
       - df_frames["birlesmis_nesne"] - df_frames["yanlis_pozitif"])
assert (chk == 0).all()

cols = ["gt_count", "pred_count", "dogru", "kacirilan", "birlesmis_hucre", "birlesmis_nesne", "yanlis_pozitif"]
print("Hata ayrıştırması (toplamlar)")
print(df_frames.groupby("split")[cols].sum().to_string())
print("\nGT hücrelerinin kategori oranları (%)")
print((pd.crosstab(df_cells["split"], df_cells["category"], normalize="index") * 100).round(1).to_string())
print("\nİşaretçi altı parlaklık (ham görüntü)")
print(df_cells.groupby(["split", "category"])["brightness"].describe()[["count", "mean", "50%"]].round(2).to_string())
print("\nSekans yoğunluk karşılaştırması (tüm kareler, Bölüm 2'den)")
print(df_overview.groupby("sequence")[["mean", "p50", "p99", "p99_9"]].mean().round(2).to_string())

# Şekil 7: hata kategorileri ve parlaklık (yalnızca grafik -> figures/)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
order = ["dogru", "birlesmis", "kacirilan"]
ct = pd.crosstab(df_cells["split"], df_cells["category"]).reindex(columns=order, fill_value=0)
ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#2E7D32", "#F9A825", "#C62828"], rot=0)
axes[0].set_ylabel("GT hücre sayısı (toplam)"); axes[0].grid(axis="y", alpha=0.3)
axes[0].set_title("GT hücrelerinin akıbeti", fontsize=10, fontweight="bold")
groups, labs = [], []
for split in ct.index:
    for cat in order:
        vals = df_cells.loc[(df_cells["split"] == split) & (df_cells["category"] == cat), "brightness"]
        if len(vals):
            groups.append(vals); labs.append(f"{split.split(' ')[0]}\n{cat}")
axes[1].boxplot(groups); axes[1].set_xticks(range(1, len(labs) + 1), labs)
axes[1].tick_params(axis="x", labelsize=7.5); axes[1].grid(alpha=0.3)
axes[1].set_ylabel("İşaretçi altı ortalama parlaklık")
axes[1].set_title("Kategoriye göre hücre parlaklığı", fontsize=10, fontweight="bold")
plt.suptitle("Şekil 7. Nihai pipeline hata ayrıştırması", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig7_error_decomposition.png", dpi=300); plt.show()

# Şekil 6: başarılı / zorlayıcı / hatalı örnekler (veri seti görüntüsü -> figures_report/)
def draw_case(ax, seq, frame_idx, fname, tag):
    cells, info, img, pred_lbl, gt = analyze_frame(seq, frame_idx, fname, FINAL_CFG)
    hi = np.percentile(img, 99.8)
    vis = cv2.cvtColor(np.clip(img * (255.0 / max(hi, 1)), 0, 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    cs, _ = cv2.findContours((pred_lbl > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(vis, cs, -1, (0, 200, 255), 2)
    col = {"dogru": (60, 220, 60), "birlesmis": (255, 210, 0), "kacirilan": (255, 40, 40)}
    for c in cells:
        ys, xs = np.nonzero(gt == c["gt_id"])
        cv2.circle(vis, (int(xs.mean()), int(ys.mean())), 9, col[c["category"]], -1)
    ax.imshow(vis); ax.axis("off")
    ax.set_title(f"{tag}\n{fname} | GT {info['gt_count']}, tahmin {info['pred_count']}\n"
                 f"kaçırılan {info['kacirilan']}, birleşmiş {info['birlesmis_hucre']}, YP {info['yanlis_pozitif']}",
                 fontsize=8.5, fontweight="bold")

val_err = (np.array(df_val[df_val["Deney_ID"] == FINAL_ID].iloc[0]["Val_Tahminler"]) - val_df["gt_count"].to_numpy())
best_val = val_df.iloc[int(np.argmin(np.abs(val_err)))]
srt = df_test.sort_values("final_abs_err").reset_index(drop=True)
mid_test, worst_test = srt.iloc[len(srt) // 2], srt.iloc[-1]

fig, axes = plt.subplots(1, 3, figsize=(16, 6.2))
draw_case(axes[0], "01", best_val["frame_idx"], best_val["filename"], "Başarılı (doğrulama)")
draw_case(axes[1], "02", mid_test["frame_idx"], mid_test["filename"], "Zorlayıcı (test, medyan hata)")
draw_case(axes[2], "02", worst_test["frame_idx"], worst_test["filename"], "En hatalı (test)")
plt.suptitle("Şekil 6. Mavi: tahmin konturu | Yeşil: doğru | Sarı: birleşmiş | Kırmızı: kaçırılan (TRA işaretçileri)",
             fontsize=10.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{FIG_IMG_DIR}/fig6_success_failure_cases.png", dpi=300); plt.show()

## 14. README, requirements.txt ve .gitignore

In [ ]:
def pkg_version(*names):
    for n in names:
        try:
            return n, version(n)
        except PackageNotFoundError:
            pass
    return None, None

req = []
for names in [("numpy",), ("pandas",), ("opencv-python", "opencv-python-headless", "opencv-contrib-python"),
              ("scipy",), ("scikit-image",), ("matplotlib",)]:
    n, v = pkg_version(*names)
    if n:
        req.append(f"{n}=={v}")
with open("requirements.txt", "w") as f:
    f.write("\n".join(req) + "\n")

with open(".gitignore", "w") as f:
    f.write("data/\n*.zip\nfigures_report/\n.ipynb_checkpoints/\n__pycache__/\n")

fs = df_summary.set_index("Yontem")
readme = f'''# Klasik Görüntü İşleme ile Hücre Segmentasyonu ve Sayımı (Fluo-N2DH-GOWT1)

Cell Tracking Challenge **Fluo-N2DH-GOWT1** eğitim verisinde klasik görüntü işleme yöntemleriyle
hücre segmentasyonu, sayımı ve zamansal sayım analizi. Derin öğrenme kullanılmamıştır.

## Veri seti
- Resmî indirme bağlantısı: {DATA_URL}
- Notebook arşivi otomatik olarak indirip `data/` klasörüne açar. **Veri seti bu repoda yer almaz.**

## Tekrarlanabilirlik
- Random seed: **{SEED}** (öğrenci numarası son 5 hane {STUDENT_ID_LAST5} + 2026, aritmetik toplam)
- Zamansal boşluk kuralı: iki kare arası fark > {MIN_GAP}
- Seçilen kareler: `selected_frames.csv`

## Kurulum ve çalıştırma
1. Google Colab'da `main.ipynb` dosyasını açın (yerelde: `pip install -r requirements.txt`).
2. *Çalışma zamanı → Tümünü çalıştır.* Veri seti indirilir, tüm deneyler, tablolar ve şekiller üretilir.
3. Bölümler sırasıyla çalışır: veri inceleme → seçim → referanslar → deneyler → parametre taramaları →
   doğrulama → **kilitleme** → test değerlendirmesi → zamansal analiz → şekiller → hata analizi.

## Nihai pipeline
**{FINAL_NAME}** (`{FINAL_ID}`): {" → ".join(s.replace(chr(10), " ") for s in flow_steps(FINAL_CFG))}

Seçim, en düşük doğrulama MAE değerine göre otomatik yapılmıştır (`validation_results.csv`).
Parametreler `final_pipeline.json` dosyasındadır.

| Yöntem | Test MAE | Test RMSE | Bias | Exact (%) |
|---|---|---|---|---|
''' + "\n".join(f"| {k} | {r['MAE']} | {r['RMSE']} | {r['Bias']} | {r['Exact_Accuracy_%']} |" for k, r in fs.iterrows()) + '''

## Test güvenliği
Nihai pipeline ve test tahminleri (`final_pipeline.json`, `test_predictions_locked.csv`), test referansları
okunmadan ayrı bir commit ile kaydedilmiştir. Test değerlendirmesi sonraki commit'tedir.
Kod, sekans 02 referanslarının kilit dosyası oluşmadan okunmasını engeller.

## Üretilen dosyalar
| Dosya | İçerik |
|---|---|
| `data_overview.csv` | Tüm karelerin boyut, tip ve yoğunluk istatistikleri |
| `selected_frames.csv` | Seçilen kareler, gruplar, test blokları ve seed (referans sayısı içermez) |
| `dev_val_references.csv` | Geliştirme ve doğrulama referans hücre sayıları |
| `experiments.csv` / `experiments_per_frame.csv` | 20 kontrollü deney (Tablo 2) ve kare bazlı tahminler |
| `parameter_sweeps.csv` | Parametre duyarlılık analizi (geliştirme kümesi) |
| `validation_results.csv` | Üç aday pipeline doğrulama sonuçları (Tablo 3) |
| `final_pipeline.json` | Nihai pipeline ve baseline konfigürasyonu |
| `test_predictions_locked.csv` | Referanslar okunmadan kaydedilen test tahminleri |
| `test_results.csv` / `test_summary.csv` | Test sonuçları (Tablo 4) ve özet metrikler (Tablo 5) |
| `validation_vs_test.csv` | Doğrulama ve test karşılaştırması |
| `temporal_analysis.csv` | Ardışık karelerdeki sayım değişimleri |
| `error_analysis_*.csv` | Hücre ve kare bazında hata ayrıştırması |
| `figures/` | Veri seti görüntüsü içermeyen grafikler |

Veri seti görüntüsü içeren şekiller (`figures_report/`) yalnızca teknik raporda yer alır ve repoya eklenmez.
'''
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print(open("requirements.txt").read())
print("README.md, requirements.txt ve .gitignore yazıldı.")
print("Hatırlatma: GitHub'a yüklemeden önce Düzenle → Tüm çıktıları temizle.")